# Essential Imports and Configurations

In [61]:
import pandas as pd
import sqlalchemy as sqla
import os
import re

from pathlib import Path
from sqlalchemy import create_engine, text

from dotenv import load_dotenv
from typing import Dict, Optional

In [ ]:
load_dotenv()

user: str = os.getenv("DB_USER")
password: str = os.getenv("DB_PASSWORD")
host: str = os.getenv("DB_HOST")
port: str = os.getenv("PORT")
database: str = os.getenv("DB_NAME")

In [ ]:
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', 1000)         # Prevents wrapping

In [18]:
def get_connection():
    engine = create_engine(
        f"postgresql://{user}:{password}@{host}:{port}/{database}"
    )
    return engine

try:
    engine = get_connection()
    print(f"Connection to the {host} for user {user} created successfully.")

except Exception as ex:
    print("Connection could not be made due to the following error:\n", ex)

Connection to the localhost for user postgres created successfully.


In [19]:
def ingest_data() -> None:
    """
    Populates the Cyber Databoard database with the CSV files data.
    """
    engine: sqla.Engine = get_connection()
    parent: str = (Path.cwd())
    path: Path = Path(f"{parent}/data/")
    day_regex: str = r"[\w\-.]+(?=-[Ww]orkingHours)"
    type_regex: str = r"(?<=-[Ww]orkingHours-)[\w\-.]+(?=.pcap)"

    for file_name in path.iterdir():
        with open(file_name, mode='r') as file:
            datafile: pd.DataFrame = pd.read_csv(file)
            day_match: re.Match = re.search(day_regex, str(file_name.name))
            type_match: re.Match = re.search(type_regex, str(file_name))
            if day_match:
                day_match: str = day_match.group().replace("_", " ")
            if type_match:
                type_match: str = type_match.group().replace("_", " ")
            else:
                type_match: str = "Working_Hours"

            table_name: str = f"{day_match}_{type_match}".replace(" ", "_")
            
            datafile.to_sql(
                f"{table_name}",
                engine, 
                if_exists="replace",
                index=False
            )


# Data Analysis

In [64]:
def query_database(query: str, para_bindings: Optional[Dict[str, str | int]] = None) -> pd.DataFrame:
    with engine.connect() as connection:
        query: sqla.Text = text(query)

        result: sqla.CursorResult = connection.execute(query, para_bindings)
        df: pd.DataFrame = pd.DataFrame(result)
        
        return df

# Friday_DDOS Analysis

In [ ]:
Friday_DDOS: pd.DataFrame = query_database('SELECT * FROM "Friday_Afternoon-DDos"')

In [70]:
Friday_DDOS.info()

<class 'pandas.DataFrame'>
RangeIndex: 225745 entries, 0 to 225744
Data columns (total 79 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0    Destination Port             225745 non-null  int64  
 1    Flow Duration                225745 non-null  int64  
 2    Total Fwd Packets            225745 non-null  int64  
 3    Total Backward Packets       225745 non-null  int64  
 4   Total Length of Fwd Packets   225745 non-null  int64  
 5    Total Length of Bwd Packets  225745 non-null  int64  
 6    Fwd Packet Length Max        225745 non-null  int64  
 7    Fwd Packet Length Min        225745 non-null  int64  
 8    Fwd Packet Length Mean       225745 non-null  float64
 9    Fwd Packet Length Std        225745 non-null  float64
 10  Bwd Packet Length Max         225745 non-null  int64  
 11   Bwd Packet Length Min        225745 non-null  int64  
 12   Bwd Packet Length Mean       225745 non-null  float64


In [71]:
Friday_DDOS.describe()

C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,225745.00000,2.257450e+05,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,...,225745.000000,225745.000000,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05
mean,8879.61946,1.624165e+07,4.874916,4.572775,939.463346,5.960477e+03,538.535693,27.882221,164.826715,214.907242,...,3.311497,21.482753,1.848261e+05,1.293436e+04,2.080849e+05,1.776201e+05,1.032214e+07,3.611943e+06,1.287813e+07,7.755355e+06
std,19754.64740,3.152437e+07,15.422874,21.755356,3249.403484,3.921834e+04,1864.128991,163.324159,504.892965,797.411073,...,12.270018,4.166799,7.979250e+05,2.102737e+05,9.002350e+05,7.842602e+05,2.185303e+07,1.275689e+07,2.692126e+07,1.983109e+07
min,0.00000,-1.000000e+00,1.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,80.00000,7.118000e+04,2.000000,1.000000,26.000000,0.000000e+00,6.000000,0.000000,6.000000,0.000000,...,1.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,80.00000,1.452333e+06,3.000000,4.000000,30.000000,1.640000e+02,20.000000,0.000000,8.666667,5.301991,...,2.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,80.00000,8.805237e+06,5.000000,5.000000,63.000000,1.160100e+04,34.000000,6.000000,32.000000,10.263203,...,4.000000,20.000000,1.878000e+03,0.000000e+00,1.878000e+03,1.862000e+03,8.239725e+06,0.000000e+00,8.253838e+06,7.422849e+06
max,65532.00000,1.199999e+08,1932.000000,2942.000000,183012.000000,5.172346e+06,11680.000000,1472.000000,3867.000000,6692.644993,...,1931.000000,52.000000,1.000000e+08,3.950000e+07,1.000000e+08,1.000000e+08,1.200000e+08,6.530000e+07,1.200000e+08,1.200000e+08


In [ ]:
Friday_DDOS.isnull().any()

 Destination Port               False
 Flow Duration                  False
 Total Fwd Packets              False
 Total Backward Packets         False
Total Length of Fwd Packets     False
 Total Length of Bwd Packets    False
 Fwd Packet Length Max          False
 Fwd Packet Length Min          False
 Fwd Packet Length Mean         False
 Fwd Packet Length Std          False
Bwd Packet Length Max           False
 Bwd Packet Length Min          False
 Bwd Packet Length Mean         False
 Bwd Packet Length Std          False
Flow Bytes/s                     True
 Flow Packets/s                 False
 Flow IAT Mean                  False
 Flow IAT Std                   False
 Flow IAT Max                   False
 Flow IAT Min                   False
Fwd IAT Total                   False
 Fwd IAT Mean                   False
 Fwd IAT Std                    False
 Fwd IAT Max                    False
 Fwd IAT Min                    False
Bwd IAT Total                   False
 Bwd IAT Mea

In [102]:
Friday_DDOS['Flow Bytes/s'].isnull().sum()

# Since there are so little missing values, we can just drop them

print(f"Before removing missing values: {Friday_DDOS.shape}")
Friday_DDOS.dropna(inplace=True)
print(f"After removing missing values: {Friday_DDOS.shape}")

Before removing missing values: (225745, 79)
After removing missing values: (225741, 79)
